In [1]:
pip install mysql-connector-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mysql.connector

In [3]:
# Database Connection

con = mysql.connector.connect(
    host="localhost",
    user="root",
    password="goutham",
    database="hospital_db"
)

cursor = con.cursor(buffered=True)

print("Database Connected Successfully")
logged_in_patient_id = None

Database Connected Successfully


In [4]:
# Main Menu Function

def main_menu():
    print("\n")
    print("=" * 50)
    print(" HOSPITAL APPOINTMENT BOOKING SYSTEM ")
    print("=" * 50)
    print("1. Admin Login")
    print("2. Patient Registration")
    print("3. Patient Login")
    print("4. Exit")

In [5]:
# Main Program

while True:

    main_menu()

    choice = input("Enter your choice (1-4): ")

    if choice == "1":
        admin_login()

    elif choice == "2":
        patient_registration()

    elif choice == "3":
        patient_login()

    elif choice == "4":
        print("Thank You!")
        break

    else:
        print("Invalid Choice!")



 HOSPITAL APPOINTMENT BOOKING SYSTEM 
1. Admin Login
2. Patient Registration
3. Patient Login
4. Exit
Thank You!


In [6]:
# Admin Login Function

def admin_login():

    username = input("Enter Admin Username : ")
    password = input("Enter Admin Password : ")

    query = "SELECT * FROM admin WHERE username=%s AND password=%s"

    cursor.execute(query, (username, password))

    admin = cursor.fetchone()

    if admin:
        print("\nLogin Successful!")
        admin_menu()

    else:
        print("\nInvalid Username or Password!")

In [7]:
# Admin Menu Function

def admin_menu():

    while True:

        print("\n")
        print("="*40)
        print("          ADMIN MENU")
        print("="*40)
        print("1. Add Doctor")
        print("2. View Doctors")
        print("3. Delete Doctor")
        print("4. View Patients")
        print("5. View All Appointments")
        print("6. Logout")

        choice = input("Enter your choice: ")

        if choice == "1":
            add_doctor()

        elif choice == "2":
            view_doctors()

        elif choice == "3":
            delete_doctor()

        elif choice == "4":
            view_patients()

        elif choice == "5":
            view_all_appointments()

        elif choice == "6":
            print("Logged Out Successfully!")
            break

        else:
            print("Invalid Choice!")

In [8]:
def add_doctor():

    print("\n========== ADD DOCTOR ==========")

    doctor_name = input("Enter Doctor Name : ")
    specialization = input("Enter Specialization : ")
    phone = input("Enter Phone Number : ")
    email = input("Enter Email : ")

    query = """
    INSERT INTO doctors
    (doctor_name, specialization, phone, email)
    VALUES (%s, %s, %s, %s)
    """

    values = (doctor_name, specialization, phone, email)

    cursor.execute(query, values)
    con.commit()

    print("\nDoctor Added Successfully!")

In [9]:
# View Doctors

def view_doctors():

    query = "SELECT * FROM doctors"

    cursor.execute(query)

    doctors = cursor.fetchall()

    if doctors:

        print("\n========== DOCTORS LIST ==========\n")

        print("ID\tName\t\tSpecialization\t\tPhone")

        for doctor in doctors:

            print(f"{doctor[0]}\t{doctor[1]}\t\t{doctor[2]}\t\t{doctor[3]}")

    else:

        print("No Doctors Found!")

In [10]:
# Patient Registration Function

def patient_registration():

    print("\n========== PATIENT REGISTRATION ==========")

    name = input("Enter Patient Name : ")
    age = int(input("Enter Age : "))
    gender = input("Enter Gender : ")
    phone = input("Enter Phone Number : ")
    email = input("Enter Email : ")
    password = input("Create Password : ")

    query = """
    INSERT INTO patients
    (name, age, gender, phone, email, password)
    VALUES (%s, %s, %s, %s, %s, %s)
    """

    values = (name, age, gender, phone, email, password)

    cursor.execute(query, values)
    con.commit()

    print("\nPatient Registered Successfully!")

In [11]:
# Patient Login Function

def patient_login():

    print("\n========== PATIENT LOGIN ==========")

    email = input("Enter Email : ")
    password = input("Enter Password : ")

    query = """
    SELECT * FROM patients
    WHERE email=%s AND password=%s
    """

    values = (email, password)

    cursor.execute(query, values)

    patient = cursor.fetchone()

    
    if patient:

     global logged_in_patient_id

     logged_in_patient_id = patient[0]

     print("\nLogin Successful!")
     patient_menu()

    else:
     print("\nInvalid Email or Password!")

In [12]:
# Patient Menu

def patient_menu():

    while True:

        print("\n")
        print("="*40)
        print("         PATIENT MENU")
        print("="*40)
        print("1. Book Appointment")
        print("2. View My Appointments")
        print("3. search doctor by specialization")
        print("4. cancel Appointments")
        print("5. Logout")
    

        choice = input("Enter your choice : ")

        if choice == "1":
            book_appointment()

        elif choice == "2":
            view_my_appointments()
        
        elif choice == "3":
            search_doctor_by_specialization()

        elif choice == "4":
            cancel_appointment()


        elif choice == "5":
            print("Logged Out Successfully!")
            break

        else:
            print("Invalid Choice!")

In [13]:
def book_appointment():

    print("\n========== BOOK APPOINTMENT ==========")

    # Available Doctors
    cursor.execute("SELECT * FROM doctors")

    doctors = cursor.fetchall()

    print("\nAvailable Doctors\n")

    for doctor in doctors:
        print("Doctor ID :", doctor[0])
        print("Name      :", doctor[1])
        print("Speciality:", doctor[2])
        print("-----------------------------")

    patient_id = logged_in_patient_id
    doctor_id = int(input("Enter Doctor ID : "))
    appointment_date = input("Enter Appointment Date (YYYY-MM-DD) : ")
    appointment_time = input("Enter Appointment Time (HH:MM) : ")

    status = "Pending"

    query = """
    INSERT INTO appointments
    (patient_id, doctor_id, appointment_date, appointment_time, status)
    VALUES (%s,%s,%s,%s,%s)
    """

    values = (
        patient_id,
        doctor_id,
        appointment_date,
        appointment_time,
        status
    )

    cursor.execute(query, values)

    con.commit()

    print("\nAppointment Booked Successfully!")

In [14]:
def view_my_appointments():

    patient_id = logged_in_patient_id
    query = """
    SELECT appointment_id,
           doctor_id,
           appointment_date,
           appointment_time,
           status
    FROM appointments
    WHERE patient_id=%s
    """

    cursor.execute(query, (patient_id,))

    appointments = cursor.fetchall()

    if appointments:

        print("\n===== MY APPOINTMENTS =====")

        for app in appointments:

            print("Appointment ID :", app[0])
            print("Doctor ID      :", app[1])
            print("Date           :", app[2])
            print("Time           :", app[3])
            print("Status         :", app[4])
            print("-------------------------")

    else:
        print("No Appointments Found")

In [15]:
def delete_doctor():

    print("\n========== DELETE DOCTOR ==========")

    doctor_id = int(input("Enter Doctor ID to Delete : "))

    query = "DELETE FROM doctors WHERE doctor_id=%s"

    cursor.execute(query, (doctor_id,))

    con.commit()

    if cursor.rowcount > 0:
        print("Doctor Deleted Successfully!")
    else:
        print("Doctor ID Not Found!")

In [16]:
def view_patients():

    print("\n========== PATIENTS LIST ==========")

    cursor.execute("SELECT * FROM patients")

    patients = cursor.fetchall()

    if patients:

        for patient in patients:

            print("Patient ID :", patient[0])
            print("Name       :", patient[1])
            print("Age        :", patient[2])
            print("Gender     :", patient[3])
            print("Phone      :", patient[4])
            print("Email      :", patient[5])
            print("-----------------------------------")

    else:
        print("No Patients Found!")

In [17]:
def view_all_appointments():

    print("\n========== ALL APPOINTMENTS ==========\n")

    query = """
    SELECT
        a.appointment_id,
        p.name,
        d.doctor_name,
        a.appointment_date,
        a.appointment_time,
        a.status
    FROM appointments a
    JOIN patients p
        ON a.patient_id = p.patient_id
    JOIN doctors d
        ON a.doctor_id = d.doctor_id
    """

    cursor.execute(query)

    appointments = cursor.fetchall()

    if appointments:

        for app in appointments:

            print("Appointment ID :", app[0])
            print("Patient Name   :", app[1])
            print("Doctor Name    :", app[2])
            print("Date           :", app[3])
            print("Time           :", app[4])
            print("Status         :", app[5])
            print("--------------------------------------")

    else:
        print("No Appointments Found!")

In [18]:
def search_doctor_by_specialization():

    print("\n========== SEARCH DOCTOR ==========")

    specialization = input("Enter Specialization : ")

    query = """
    SELECT doctor_id, doctor_name, specialization, phone, email
    FROM doctors
    WHERE specialization=%s
    """

    cursor.execute(query, (specialization,))

    doctors = cursor.fetchall()

    if doctors:

        print("\nAvailable Doctors\n")

        for doctor in doctors:

            print("Doctor ID      :", doctor[0])
            print("Doctor Name    :", doctor[1])
            print("Specialization :", doctor[2])
            print("Phone          :", doctor[3])
            print("Email          :", doctor[4])
            print("-----------------------------------")

    else:
        print("\nNo Doctor Found with this Specialization.")

In [19]:
def cancel_appointment():

    patient_id = logged_in_patient_id

    appointment_id = int(input("Enter Appointment ID to Cancel : "))

    query = """
    UPDATE appointments
    SET status='Cancelled'
    WHERE appointment_id=%s AND patient_id=%s
    """

    cursor.execute(query, (appointment_id, patient_id))
    con.commit()

    if cursor.rowcount > 0:
        print("\nAppointment Cancelled Successfully!")
    else:
        print("\nAppointment Not Found!")
        